# 00 · Pre-config — RetailHub setup

Trainer-only notebook. It creates the per-user course catalog, schemas and
Volume used by the rest of `Databricks-Data-Analyst-Medi`.

Run this once before the course for every participant catalog pattern you need.

Before running, confirm `STORAGE_LOCATION` points to a valid Unity Catalog
managed storage location for this workspace.

In [ ]:
import re

CATALOG_PREFIX = "retailhub"

# Managed location for catalogs. This follows the Data Engineer Associate setup pattern.
# Example Azure: "abfss://container@account.dfs.core.windows.net/path"
# Leave empty only when the metastore default managed location is configured.
STORAGE_LOCATION = "abfss://training@dbxaltextstorage.dfs.core.windows.net/"

SCHEMAS = ["default", "bronze", "silver", "gold"]
VOLUME_NAME = "datasets"

def resolve_user_slug(current_user):
    identity = current_user.lower()
    local_part = identity.split("@")[0]

    if "trainer" in identity or "trener" in identity or local_part == "krzysztof.burejza":
        return "trainer"

    user_slug = re.sub(r"[^a-zA-Z0-9]", "_", local_part).lower()
    return re.sub(r"_+", "_", user_slug).strip("_") or "user"

current_user = spark.sql("SELECT current_user()").first()[0]
user_slug = resolve_user_slug(current_user)

CATALOG = f"{CATALOG_PREFIX}_{user_slug}"
MANAGED_LOCATION = f"{STORAGE_LOCATION.rstrip('/')}/{CATALOG}" if STORAGE_LOCATION else ""

print("Provisioning catalog:", CATALOG)
print("Managed location:", MANAGED_LOCATION or "(metastore default)")

In [ ]:
if MANAGED_LOCATION:
    spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS {CATALOG}
        MANAGED LOCATION '{MANAGED_LOCATION}'
    """)
else:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

spark.sql(f"USE CATALOG {CATALOG}")

for schema in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.default.{VOLUME_NAME}")

DATASET_PATH = f"/Volumes/{CATALOG}/default/{VOLUME_NAME}"
print("Dataset Volume:", DATASET_PATH)
print("[OK] Catalog, schemas and Volume are ready.")

## Next step

Run `data/generate_training_dataset.ipynb`. The generator creates synthetic
RetailHub data, including controlled data-quality issues for the workshops.